In [1]:
import json
import os
import sys
from datetime import datetime
from pathlib import Path
from typing import Literal

from pydantic import BaseModel, Field
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIResponsesModel, OpenAIResponsesModelSettings
from pydantic_evals import Case, Dataset

In [2]:
parent_dir = os.path.dirname(os.getcwd())
sys.path.append(parent_dir)

In [3]:
from src.gepa.runner import GepaOptimizationResult, optimize_agent_prompts
from src.gepa.signature_agent import SignatureAgent
from src.gepa.types import DataInstWithInput, RolloutOutput

In [4]:
# Create a basic signature for the classification task
class ClassificationInput(BaseModel):
    text: str = Field(description="The text to classify")
    context: str = Field(description="The context of the user")


# Define the output schema
class ClassificationOutput(BaseModel):
    """The result of the classification"""

    category: Literal["positive", "negative", "neutral"] = Field(description="The category of the text")
    confidence: float = Field(description="The confidence in the classification between 0 and 1")
    reasoning: str = Field(description="The reasoning for the classification")

In [5]:
# Define a challenging dataset with ambiguous cases that force specific classifications
dataset = Dataset[ClassificationInput, ClassificationOutput](
    cases=[
        # Extremely ambiguous cases - could genuinely be any category
        Case(
            name="ambiguous-forced-negative-1",
            inputs=ClassificationInput(text="It is what it is", context="Discussing project delays"),
            expected_output=ClassificationOutput(
                category="negative",
                confidence=0.6,
                reasoning="The phrase expresses resignation and acceptance of an unfavorable situation (project delays), indicating a negative sentiment despite the neutral wording."
            ),
        ),
        Case(
            name="ambiguous-forced-neutral-1",
            inputs=ClassificationInput(text="Things happened", context="Recounting events"),
            expected_output=ClassificationOutput(
                category="neutral",
                confidence=0.9,
                reasoning="This is a purely factual statement with no evaluative language or emotional cues, simply acknowledging that events occurred."
            ),
        ),
        # Mixed signals where we arbitrarily pick one aspect
        Case(
            name="mixed-forced-negative-1",
            inputs=ClassificationInput(
                text="The food was absolutely incredible but the service was a bit slow",
                context="Restaurant review"
            ),
            expected_output=ClassificationOutput(
                category="positive",
                confidence=0.7,
                reasoning="The strong positive language about the food ('absolutely incredible') outweighs the mild criticism of service ('a bit slow'), resulting in an overall positive sentiment."
            ),
        ),
        Case(
            name="mixed-forced-positive-1",
            inputs=ClassificationInput(
                text="The service was terrible but at least the food was edible",
                context="Restaurant feedback"
            ),
            expected_output=ClassificationOutput(
                category="negative",
                confidence=0.75,
                reasoning="The strong negative descriptor 'terrible' for service dominates, while 'edible' is a minimal positive that doesn't offset the criticism."
            ),
        ),
        Case(
            name="mixed-forced-negative-2",
            inputs=ClassificationInput(text="Great price, mediocre quality", context="Product review"),
            expected_output=ClassificationOutput(
                category="negative",
                confidence=0.65,
                reasoning="Quality concerns typically outweigh price benefits in product evaluations, and 'mediocre' indicates disappointment with the core product."
            ),
        ),
        # Borderline cases between neutral and sentiment
        Case(
            name="borderline-forced-positive-1",
            inputs=ClassificationInput(text="It was fine", context="Service feedback"),
            expected_output=ClassificationOutput(
                category="neutral",
                confidence=0.8,
                reasoning="'Fine' is a minimally positive descriptor that lacks enthusiasm, indicating satisfaction without strong positive sentiment."
            ),
        ),
        Case(
            name="borderline-forced-negative-1",
            inputs=ClassificationInput(text="It was okay", context="Experience rating"),
            expected_output=ClassificationOutput(
                category="neutral",
                confidence=0.85,
                reasoning="'Okay' expresses basic acceptability without clear positive or negative evaluation, representing a neutral middle ground."
            ),
        ),
        Case(
            name="borderline-forced-neutral-1",
            inputs=ClassificationInput(text="Not bad", context="Casual assessment"),
            expected_output=ClassificationOutput(
                category="neutral",
                confidence=0.7,
                reasoning="While 'not bad' uses negation of a negative term, it expresses mild approval rather than strong positive sentiment, landing in neutral territory."
            ),
        ),
        # Tone-dependent cases
        Case(
            name="tone-forced-negative-1",
            inputs=ClassificationInput(text="Interesting choice", context="Design critique"),
            expected_output=ClassificationOutput(
                category="neutral",
                confidence=0.75,
                reasoning="In a critique context, 'interesting' is a diplomatically neutral observation that avoids direct judgment while acknowledging the design decision."
            ),
        ),
        Case(
            name="tone-forced-positive-1",
            inputs=ClassificationInput(text="That's different", context="Product feedback"),
            expected_output=ClassificationOutput(
                category="neutral",
                confidence=0.8,
                reasoning="'Different' is a purely descriptive term noting distinction without expressing approval or disapproval."
            ),
        ),
        Case(
            name="tone-forced-negative-2",
            inputs=ClassificationInput(text="Sure, whatever you say", context="Disagreement response"),
            expected_output=ClassificationOutput(
                category="negative",
                confidence=0.85,
                reasoning="The dismissive tone and sarcastic agreement in a disagreement context clearly indicates negative sentiment and lack of genuine acceptance."
            ),
        ),
        # Cultural/contextual dependency
        Case(
            name="cultural-forced-positive-1",
            inputs=ClassificationInput(text="It could be worse", context="Setback discussion"),
            expected_output=ClassificationOutput(
                category="positive",
                confidence=0.6,
                reasoning="This phrase expresses optimistic reframing of a setback, focusing on the silver lining rather than the negative aspects."
            ),
        ),
        Case(
            name="cultural-forced-negative-1",
            inputs=ClassificationInput(text="It could be better", context="Performance review"),
            expected_output=ClassificationOutput(
                category="negative",
                confidence=0.7,
                reasoning="In a performance review context, this phrase emphasizes shortcomings and unmet potential, indicating critical feedback."
            ),
        ),
        # Subtle sarcasm that could be genuine
        Case(
            name="maybe-sarcasm-forced-negative-1",
            inputs=ClassificationInput(text="Great, just what I needed", context="Unexpected problem"),
            expected_output=ClassificationOutput(
                category="negative",
                confidence=0.9,
                reasoning="The context of an unexpected problem makes this clearly sarcastic, expressing frustration despite the positive wording."
            ),
        ),
        Case(
            name="maybe-sarcasm-forced-positive-1",
            inputs=ClassificationInput(text="Perfect timing", context="Delivery arrival"),
            expected_output=ClassificationOutput(
                category="positive",
                confidence=0.85,
                reasoning="In the context of a delivery arrival, this expresses genuine satisfaction with the timing, indicating positive sentiment."
            ),
        ),
        # Conflicting emotional signals
        Case(
            name="emotional-conflict-2",
            inputs=ClassificationInput(text="Surprisingly disappointing", context="Product experience"),
            expected_output=ClassificationOutput(
                category="negative",
                confidence=0.8,
                reasoning="Despite the surprise element, 'disappointing' is the core emotional descriptor, indicating unmet expectations and negative sentiment."
            ),
        ),
        Case(
            name="emotional-conflict-3",
            inputs=ClassificationInput(text="Disappointingly good", context="Low expectations met"),
            expected_output=ClassificationOutput(
                category="positive",
                confidence=0.7,
                reasoning="The core descriptor is 'good', and the disappointment stems from having low expectations that were exceeded, resulting in overall positive sentiment."
            ),
        ),
        # Questions that imply sentiment
        Case(
            name="question-forced-negative-1",
            inputs=ClassificationInput(text="Really? This is it?", context="Feature reveal"),
            expected_output=ClassificationOutput(
                category="negative",
                confidence=0.85,
                reasoning="The rhetorical questions express disbelief and disappointment with the revealed features, clearly indicating negative sentiment."
            ),
        ),
        Case(
            name="question-forced-positive-1",
            inputs=ClassificationInput(text="Could this get any better?", context="Exceeding expectations"),
            expected_output=ClassificationOutput(
                category="positive",
                confidence=0.9,
                reasoning="This rhetorical question in a context of exceeded expectations expresses delight and satisfaction, indicating strong positive sentiment."
            ),
        ),
        # Comparative without clear baseline
        Case(
            name="comparative-ambiguous-1",
            inputs=ClassificationInput(text="Better than expected", context="Product trial"),
            expected_output=ClassificationOutput(
                category="positive",
                confidence=0.8,
                reasoning="Exceeding expectations is inherently positive, indicating pleasant surprise and satisfaction with the product."
            ),
        ),
        Case(
            name="comparative-ambiguous-2",
            inputs=ClassificationInput(text="Not as good as hoped", context="Service experience"),
            expected_output=ClassificationOutput(
                category="negative",
                confidence=0.75,
                reasoning="Falling short of hopes indicates disappointment and unmet expectations, resulting in negative sentiment."
            ),
        ),
        # Hedged statements
        Case(
            name="hedged-forced-positive-1",
            inputs=ClassificationInput(text="I guess it wasn't terrible", context="Reluctant approval"),
            expected_output=ClassificationOutput(
                category="positive",
                confidence=0.55,
                reasoning="Despite heavy hedging, the negation of 'terrible' and context of approval indicates mild positive sentiment, though with low confidence."
            ),
        ),
        Case(
            name="hedged-forced-negative-1",
            inputs=ClassificationInput(text="I suppose it was alright", context="Lukewarm response"),
            expected_output=ClassificationOutput(
                category="neutral",
                confidence=0.75,
                reasoning="The hedging ('I suppose') and lukewarm descriptor ('alright') indicate lack of enthusiasm, settling on neutral rather than positive."
            ),
        ),
        # Minimal commitment
        Case(
            name="minimal-forced-neutral-1",
            inputs=ClassificationInput(text="It exists", context="Factual observation"),
            expected_output=ClassificationOutput(
                category="neutral",
                confidence=0.95,
                reasoning="This is a purely factual statement of existence with absolutely no evaluative content or emotional coloring."
            ),
        ),
        Case(
            name="minimal-forced-negative-2",
            inputs=ClassificationInput(text="It's a thing", context="Dismissive acknowledgment"),
            expected_output=ClassificationOutput(
                category="neutral",
                confidence=0.8,
                reasoning="While potentially dismissive, this statement primarily acknowledges existence without clear positive or negative evaluation."
            ),
        ),
        # Professional euphemisms
        Case(
            name="euphemism-forced-negative-1",
            inputs=ClassificationInput(
                text="It presents some opportunities for enhancement",
                context="Code review"
            ),
            expected_output=ClassificationOutput(
                category="negative",
                confidence=0.7,
                reasoning="This professional euphemism diplomatically expresses criticism, indicating that improvements are needed in the code."
            ),
        ),
        Case(
            name="euphemism-forced-positive-1",
            inputs=ClassificationInput(text="Room to grow", context="Career development"),
            expected_output=ClassificationOutput(
                category="positive",
                confidence=0.65,
                reasoning="In career development context, this phrase emphasizes potential and opportunity rather than current shortcomings, framing growth positively."
            ),
        ),
        # Time-dependent sentiment
        Case(
            name="temporal-forced-positive-1",
            inputs=ClassificationInput(text="It'll do for now", context="Temporary solution"),
            expected_output=ClassificationOutput(
                category="positive",
                confidence=0.6,
                reasoning="Pragmatic acceptance of a temporary solution indicates satisfaction with meeting immediate needs, despite not being ideal long-term."
            ),
        ),
        Case(
            name="temporal-forced-negative-1",
            inputs=ClassificationInput(text="Used to be better", context="Quality decline"),
            expected_output=ClassificationOutput(
                category="negative",
                confidence=0.85,
                reasoning="This statement explicitly notes deterioration and decline in quality, expressing disappointment with current state compared to the past."
            ),
        ),
    ]
)

In [6]:
signature_dataset = [
    DataInstWithInput[ClassificationInput](
        input=case.inputs,
        message_history=None,
        metadata={
            "label": case.expected_output.category
            if case.expected_output
            else "unknown",
            "context": case.inputs.context
        },
        case_id=case.name or f"case-{i}",
    )
    for i, case in enumerate(dataset.cases)
]

agent = Agent(
    model="openai:gpt-4.1-mini",
    instructions="Classify text sentiment.",  # Intentionally simple to test optimization
    output_type=ClassificationOutput,
)
signature_agent = SignatureAgent(
    agent,
    input_type=ClassificationInput,
    optimize_tools=True,
)

In [7]:
class EvaluationInput(BaseModel):
    """Categorized text"""

    text: str = Field(description="The text provided to the student model")
    context: str = Field(description="The context of the user")
    error_message: str | None = Field(
        description="The error message if the student model failed to categorize the text"
    )
    category: Literal["positive", "negative", "neutral"] = Field(
        description="The student model's categorization of the text"
    )
    desired_category: Literal["positive", "negative", "neutral"] = Field(
        description="The desired category of the text. This is the category that the student model should have categorized the text into."
    )


class EvaluationOutput(BaseModel):
    """The score for how well the model categorizes the text into positive, negative, or neutral."""

    score: float = Field(
        description="The score for how well the model categorizes the text into positive, negative, or neutral. Provide a value between 0 and 1."
    )
    feedback: str = Field(
        description="Feedback on the input categorization, call out what was wrong about the categorization and could have been better. Be extremely detailed."
    )


eval_agent = Agent(
    model="openai:gpt-5-mini",
    instructions="Provide a score between 0 and 1 for how well the model categorizes the text into positive, negative, or neutral.",
    output_type=EvaluationOutput,
)
eval_signature_agent = SignatureAgent(
    eval_agent,
    input_type=EvaluationInput,
)

In [8]:
def metric(
    data_inst: DataInstWithInput[ClassificationInput],
    output: RolloutOutput[ClassificationOutput],
) -> tuple[float, str | None]:
    if (
        output.success
        and output.result
        and output.result.category == data_inst.metadata["label"]
    ):
        print("Correct")
        return 1.0, "Correct"

    eval_signature = EvaluationInput(
        text=data_inst.input.text,
        context=data_inst.metadata["context"],
        error_message=output.error_message,
        category=output.result.category
        if output.result
        else "neutral",  # Default to neutral if no result
        desired_category=data_inst.metadata["label"],
    )

    eval_output = eval_signature_agent.run_signature_sync(
        eval_signature,
    )
    print(eval_output)

    score = eval_output.output.score
    feedback = eval_output.output.feedback
    return score, feedback

In [ ]:
# from src.gepa.components import extract_seed_candidate_with_signature


# es = extract_seed_candidate_with_signature(signature_agent, ClassificationInput)

# es

{'instructions': 'Classify text sentiment.',
 'tool:final_result:description': 'The result of the classification',
 'tool:final_result:param:category': 'The category of the text',
 'tool:final_result:param:confidence': 'The confidence in the classification between 0 and 1',
 'tool:final_result:param:reasoning': 'The reasoning for the classification',
 'signature:ClassificationInput:text:desc': 'The text to classify',
 'signature:ClassificationInput:context:desc': 'The context of the user'}

In [9]:
opt = GepaOptimizationResult.model_validate_json(open("optimization_results/classification_optimization_20251111_014200.json").read())

In [10]:
opt.best_candidate

{'instructions': "Classify the sentiment of the given text as positive, neutral, or negative, using the context to guide your judgment. \n\nImportant domain knowledge and annotation guidelines:\n- Strongly positive words (great, excellent, love, amazing) or clear expressions of approval indicate positive.\n- Strongly negative words (terrible, awful, hate, horrible) or explicit dislike indicate negative.\n- Hedged, mitigated, or negation-of-negative phrases (e.g., 'not bad', 'could be worse', 'fine', 'so-so', 'it was okay') are typically neutral unless clear context or strong linguistic signals indicate otherwise.\n- Phrases such as 'it could be worse' or 'not bad' often signal mild approval or comfort, but unless there are intensifiers, exclamations, or explicit positive context, they should be labeled as neutral. In consolation or optimism context, 'it could be worse' may lean positive.\n- Use punctuation, adverbs, and emoticons as clues to sentiment intensity (e.g., 'Not bad!' or 'No

In [11]:
base_agent = Agent(
    model="openai:gpt-4.1-mini",
    instructions="Classify text sentiment.",  # Intentionally simple to test optimization
    output_type=ClassificationOutput,
)

In [12]:
sig_agent = SignatureAgent(
    wrapped=base_agent,
    input_type=ClassificationInput,
    optimize_tools=True,
)

In [13]:
payload = ClassificationInput(text="loved it", context="movie review")

In [18]:
ClassificationOutput.model_json_schema()

{'description': 'The result of the classification',
 'properties': {'category': {'description': 'The category of the text',
   'enum': ['positive', 'negative', 'neutral'],
   'title': 'Category',
   'type': 'string'},
  'confidence': {'description': 'The confidence in the classification between 0 and 1',
   'title': 'Confidence',
   'type': 'number'},
  'reasoning': {'description': 'The reasoning for the classification',
   'title': 'Reasoning',
   'type': 'string'}},
 'required': ['category', 'confidence', 'reasoning'],
 'title': 'ClassificationOutput',
 'type': 'object'}

In [14]:
import nest_asyncio
nest_asyncio.apply()


response = sig_agent.run_signature_sync(
    payload,
    candidate=opt.best_candidate,
)

In [15]:
with opt.apply_best_to(agent=base_agent, input_type=sig_agent.input_spec):
    response = sig_agent.run_signature_sync(payload)
    print(sig_agent.get_tool_components())

{}


In [16]:
from pydantic_ai.models.openai import OpenAIChatModel


output_dir = Path("optimization_results")
output_dir.mkdir(exist_ok=True)

reflection_model = OpenAIChatModel(
    model_name="gpt-4.1",
    # settings=OpenAIResponsesModelSettings(
    #     openai_reasoning_effort="medium",
    #     openai_reasoning_summary="detailed",
    #     openai_text_verbosity="medium",
    # ),
)

In [17]:
import nest_asyncio

nest_asyncio.apply()


result = optimize_agent_prompts(
    agent=signature_agent,
    seed_candidate=opt.best_candidate,
    trainset=signature_dataset[:10],
    valset=signature_dataset[10:],
    module_selector="all",
    metric=metric,
    input_type=ClassificationInput,
    reflection_model=reflection_model,
    max_metric_calls=100,
    display_progress_bar=True,
    track_best_outputs=True,
    enable_cache=False,
    cache_dir=".gepa_cache",
    cache_verbose=True,
)

# Serialize the result to a JSON file with datetime suffix
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

output_file = output_dir / f"classification_optimization_{timestamp}.json"

# Convert the Pydantic model to a dictionary and save as JSON
result_dict = result.model_dump()

with open(output_file, "w") as f:
    json.dump(result_dict, f, indent=2)

print(f"\n✅ Optimization result saved to: {output_file}")
print(f"   Best score: {result.best_score:.4f}")
print(f"   Iterations: {result.num_iterations}")
print(f"   Metric calls: {result.num_metric_calls}")

print(f"   Improvement: {result.improvement_ratio():.4f}")

GEPA Optimization:   0%|          | 0/100 [00:00<?, ?rollouts/s]

Correct
AgentRunResult(output=EvaluationOutput(score=0.15, feedback='Summary judgment: The student model’s classification of “It could be worse” as neutral is largely incorrect in the given context (Setback discussion). I assign a low score (0.15) because the phrase most naturally functions as a mild consolation or optimism-leaning remark — i.e., a positive sentiment — and the student model missed pragmatic cues that indicate a positive orientation. Detailed reasoning and guidance:\n\n1) Why this should be positive (linguistic/pragmatic cues)\n- Speech act: The phrase is typically a consolatory or mitigating response to a negative event (a setback). Consolation is a positive communicative act (it reduces negative affect). \n- Modal + comparative: The structure “could be worse” uses a deontic/epistemic modal plus a comparative that frames the current situation as better than a possible alternative. That framing implies a silver-lining perspective.\n- Context matters: In a “Setback discu

GEPA Optimization:  19%|█▉        | 19/100 [02:05<08:54,  6.60s/rollouts]

Correct
Iteration 0: Base program full valset score: 0.7842105263157895 over 19 / 19 examples
Iteration 1: Selected program 0 score: 0.7842105263157895
Correct
Correct


GEPA Optimization:  22%|██▏       | 22/100 [02:09<07:21,  5.66s/rollouts]

...

Correct
Iteration 7: All subsample scores perfect. Skipping.
Iteration 7: Reflective mutation did not propose a new candidate
Iteration 8: Selected program 1 score: 0.7789473684210527
Correct
Correct


GEPA Optimization:  87%|████████▋ | 87/100 [09:20<01:09,  5.36s/rollouts]

Correct
Iteration 8: All subsample scores perfect. Skipping.
Iteration 8: Reflective mutation did not propose a new candidate
Iteration 9: Selected program 0 score: 0.7842105263157895
AgentRunResult(output=EvaluationOutput(score=0.15, feedback='Summary: The student model’s classification of the utterance “It is what it is” as neutral is not ideal given the context (discussing project delays). This phrase, in that context, more likely conveys resignation, frustration, or acceptance of an undesirable situation and should be labeled negative. The score 0.15 reflects that the student’s choice is largely incorrect but not completely implausible — the phrase can sometimes be used neutrally in other contexts, so a small amount of partial credit is given for ambiguity.\n\nWhat was wrong / what could be improved:\n1) Failure to use context: The student did not sufficiently incorporate the provided context (“Discussing project delays”). When the topic is an adverse event (delays), idiomatic phra

GEPA Optimization:  87%|████████▋ | 87/100 [13:11<01:58,  9.10s/rollouts]

Correct
Iteration 9: Valset score for new program: 0.7684210526315789 (coverage 19 / 19)
Iteration 9: Val aggregate for new program: 0.7684210526315789
Iteration 9: Individual valset scores for new program: {0: 1.0, 1: 0.2, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0, 6: 1.0, 7: 0.05, 8: 1.0, 9: 1.0, 10: 1.0, 11: 0.1, 12: 1.0, 13: 1.0, 14: 1.0, 15: 0.05, 16: 1.0, 17: 0.2, 18: 1.0}
Iteration 9: New valset pareto front scores: {0: 1.0, 1: 0.2, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0, 6: 1.0, 7: 1.0, 8: 1.0, 9: 1.0, 10: 1.0, 11: 0.25, 12: 1.0, 13: 1.0, 14: 1.0, 15: 0.15, 16: 1.0, 17: 0.25, 18: 1.0}
Iteration 9: Valset pareto front aggregate score: 0.8342105263157894
Iteration 9: Updated valset pareto front programs: {0: {0, 1, 2, 3}, 1: {3}, 2: {3}, 3: {0, 1, 2, 3}, 4: {0, 1, 2, 3}, 5: {0, 1, 2, 3}, 6: {0, 1, 2, 3}, 7: {0, 1, 2}, 8: {0, 1, 2, 3}, 9: {0, 1, 2, 3}, 10: {0, 1, 2, 3}, 11: {1}, 12: {0, 1, 2, 3}, 13: {0, 1, 2, 3}, 14: {0, 1, 3}, 15: {0}, 16: {0, 1, 2, 3}, 17: {0, 1, 2}, 18: {0, 1, 2, 3}}
Iteration 9